<a href="https://colab.research.google.com/github/Rollybuilds/library_management_system/blob/main/Library_Management_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

** PROJECT 2 - LIBRARY MANAGEMENT SYSTEM **

In [ ]:
import csv
from abc import ABC, abstractmethod

FILE_NAME = "books.csv"


# ---------------------------------------------------------------- Exception
class LibraryError(Exception):
    """Raised when a library action cannot be completed."""


# ------------------------------------------------- Abstract base class (abc)
class LibraryItem(ABC):
    """Base class for anything the library can lend."""

    def __init__(self, item_id, title):
        self.__item_id = item_id      # private attributes (encapsulation)
        self.__title = ""
        self.__issued_to = ""
        self.title = title            # uses the setter below for validation

    # Getters and setters
    @property
    def item_id(self):
        return self.__item_id

    @property
    def title(self):
        return self.__title

    @title.setter
    def title(self, value):
        if not value.strip():
            raise ValueError("Title cannot be empty.")
        self.__title = value.strip()

    @property
    def issued_to(self):
        return self.__issued_to

    @issued_to.setter
    def issued_to(self, name):
        self.__issued_to = name.strip()

    def is_available(self):
        return self.__issued_to == ""

    # Every child class must implement these
    @abstractmethod
    def to_row(self):
        """Return the item as a list, ready to be written to the CSV file."""

    @abstractmethod
    def __str__(self):
        """Return a readable description of the item."""


# ------------------------------------------------- Derived class (inheritance)
class Book(LibraryItem):
    def __init__(self, item_id, title, author):
        super().__init__(item_id, title)
        self.__author = author.strip() or "Unknown"

    @property
    def author(self):
        return self.__author

    def to_row(self):
        return [self.item_id, self.title, self.author, self.issued_to]

    def __str__(self):
        status = "Available" if self.is_available() else f"Issued to {self.issued_to}"
        return f"[{self.item_id}] {self.title} by {self.author} - {status}"


# ---------------------------------------------------------------- Library
class Library:
    """Manages all books and saves them to a CSV file."""

    def __init__(self, filename=FILE_NAME):
        self.__filename = filename
        self.__books = {}
        self.load_records()

    # ----- file handling
    def load_records(self):
        try:
            with open(self.__filename, newline="") as file:
                for row in csv.DictReader(file):
                    book = Book(int(row["id"]), row["title"], row["author"])
                    book.issued_to = row["issued_to"]
                    self.__books[book.item_id] = book
        except FileNotFoundError:
            print("No saved records found. Starting with an empty library.")
        except (KeyError, ValueError):
            print("Some records in the file could not be read.")

    def save_records(self):
        try:
            with open(self.__filename, "w", newline="") as file:
                writer = csv.writer(file)
                writer.writerow(["id", "title", "author", "issued_to"])
                for book in self.__books.values():
                    writer.writerow(book.to_row())
        except OSError:
            print("Error: could not save records to file.")

    # ----- library actions
    def add_book(self, title, author):
        new_id = max(self.__books, default=0) + 1
        book = Book(new_id, title, author)
        self.__books[new_id] = book
        self.save_records()
        return book

    def find_book(self, book_id):
        if book_id not in self.__books:
            raise LibraryError("No book found with that ID.")
        return self.__books[book_id]

    def search(self, keyword):
        keyword = keyword.lower()
        return [b for b in self.__books.values()
                if keyword in b.title.lower() or keyword in b.author.lower()]

    def issue_book(self, book_id, member_name):
        book = self.find_book(book_id)
        if not book.is_available():
            raise LibraryError(f"Already issued to {book.issued_to}.")
        if not member_name.strip():
            raise LibraryError("Member name cannot be empty.")
        book.issued_to = member_name
        self.save_records()

    def return_book(self, book_id):
        book = self.find_book(book_id)
        if book.is_available():
            raise LibraryError("This book was not issued.")
        book.issued_to = ""
        self.save_records()

    def available_books(self):
        return [b for b in self.__books.values() if b.is_available()]


# ---------------------------------------------------------------- Menu
def read_number(message):
    """Keep asking until the user enters a valid whole number."""
    while True:
        try:
            return int(input(message))
        except ValueError:
            print("Please enter a valid number.")


def show_books(books):
    if not books:
        print("No books to show.")
    for book in books:
        print(book)


def main():
    library = Library()

    menu = """
===== LIBRARY MANAGEMENT SYSTEM =====
1. Add book
2. Search books
3. Issue book
4. Return book
5. Display available books
6. Exit
"""

    while True:
        print(menu)
        choice = input("Enter your choice (1-6): ").strip()

        try:
            if choice == "1":
                title = input("Book title: ")
                author = input("Author: ")
                book = library.add_book(title, author)
                print(f"Book added with ID {book.item_id}.")

            elif choice == "2":
                show_books(library.search(input("Search by title or author: ")))

            elif choice == "3":
                book_id = read_number("Book ID: ")
                member = input("Member name: ")
                library.issue_book(book_id, member)
                print("Book issued successfully.")

            elif choice == "4":
                library.return_book(read_number("Book ID: "))
                print("Book returned successfully.")

            elif choice == "5":
                show_books(library.available_books())

            elif choice == "6":
                library.save_records()
                print("Records saved. Goodbye!")
                break

            else:
                print("Invalid choice. Please select 1-6.")

        except (LibraryError, ValueError) as error:
            print(f"Error: {error}")


if __name__ == "__main__":
    main()


===== LIBRARY MANAGEMENT SYSTEM =====
1. Add book
2. Search books
3. Issue book
4. Return book
5. Display available books
6. Exit

Enter your choice (1-6): 1
Book title: Java with OOPS
Author: Rolly
Book added with ID 5.

===== LIBRARY MANAGEMENT SYSTEM =====
1. Add book
2. Search books
3. Issue book
4. Return book
5. Display available books
6. Exit

Enter your choice (1-6): 2
Search by title or author: Java with Oops
[2] Java with OOPS by Nadeem bhajpayee - Available
[5] Java with OOPS by Rolly - Available

===== LIBRARY MANAGEMENT SYSTEM =====
1. Add book
2. Search books
3. Issue book
4. Return book
5. Display available books
6. Exit

Enter your choice (1-6): 3
Book ID: 3
Member name: Kajal
Book issued successfully.

===== LIBRARY MANAGEMENT SYSTEM =====
1. Add book
2. Search books
3. Issue book
4. Return book
5. Display available books
6. Exit

Enter your choice (1-6): 4
Book ID: 3
Book returned successfully.

===== LIBRARY MANAGEMENT SYSTEM =====
1. Add book
2. Search books
3. Iss